In [ ]:
import torch

In [ ]:
# Load Pride and Prejudice from Project Gutenberg and convert to word-based token
import urllib.request
import re

url = "https://www.gutenberg.org/cache/epub/1342/pg1342.txt"
response = urllib.request.urlopen(url)
raw_text = response.read().decode('utf-8-sig')

# Find where the actual book starts (after Gutenberg header)
# Look for "It is a truth universally acknowledged" - the famous opening line
start = raw_text.find("It is a truth universally acknowledged")
if start == -1:
    start = 0

# Find where the book ends
end = raw_text.find("*** END OF THE PROJECT GUTENBERG EBOOK")
if end == -1:
    end = len(raw_text)

text = raw_text[start:end].strip()

# Remove illustration markers and extra whitespace
text = re.sub(r'\[Illustration[^\]]*\]', '', text)
text = re.sub(r'\s+', ' ', text)
# Remove publisher info at end
text = text[0:-80]


In [34]:
# Model Hyperparams
vocab = sorted(list(set(text))) + ["<UNK>", "<PAD>"]
d_model = 256
seq_len = 256
n_heads = 4
n_layers = 4


In [ ]:
def create_causal_mask(seq_len, device):
    """Lower triangular mask for GPT self-attention"""

    # Hides output ahead of the current place in sequence so it cannot cheat
    return torch.tril(torch.ones(seq_len, seq_len, device=device)).bool()

In [ ]:
class Transformer(torch.nn.Module):
    def __init__(self, d_model, vocab, seq_len, n_heads, n_layers, d_ff, dropout):
        super().__init__()

        self.d_model = d_model
        self.vocab = vocab
        self.seq_len = seq_len
        self.n_heads = n_heads
        self.n_layers = n_layers
        self.d_ff = d_ff
        self.dropout = dropout
        self.vocab_size = len(self.vocab)

        #Mappings
        self.itos = {i: vocab[i] for i in range(len(vocab))}  # index to string
        self.stoi = {vocab[i]: i for i in range(len(vocab))}  # string to index

        #Embeddings
        self.token_embedding_layer = torch.nn.Embedding(num_embeddings=self.vocab_size, embedding_dim=self.d_model)
        self.pos_embedding_layer = torch.nn.Embedding(num_embeddings=self.seq_len, embedding_dim=self.d_model)
    
    def encode(self, text):
        """Convert text string to list of token IDs"""
        return [self.stoi[ch] for ch in text]

    def decode(self, token_ids):
        """Convert list of token IDs back to text string"""
        return ''.join([self.itos[i] for i in token_ids])
    
    def forward_pass():
        # 
        pass


In [ ]:
class MultiHeadAttention(torch.nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()

        self.d_model = d_model
        self.n_heads = n_heads
        self.dropout = dropout
        self.dk = self.d_model // self.n_heads
        # TODO: 
        self.W_q = torch.nn.Linear(self.d_model, self.d_model)
        self.W_k = torch.nn.Linear(self.d_model, self.d_model)
        self.W_v = torch.nn.Linear(self.d_model, self.d_model)
        self.O = torch.nn.Linear(self.d_model, self.d_model)
        self.dropout = torch.nn.Dropout(dropout)
        # - Dropout layer
        
    def forward(self, x, mask=None):
        """Performs The forward pass of the Multi-Head Attention
           Takes in an embedding x and returns attention vals
        """
        batch_size, seq_len, _ = x.shape

        # Linear transformation to Query, Key, Value vals
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        # Split into heads and realign for efficient parallel calc
        Q = Q.view(batch_size, seq_len, self.n_heads, self.dk).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.n_heads, self.dk).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.n_heads, self.dk).transpose(1, 2)

        context = self.attention(Q, K, V, mask)
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        return self.O(context)

    def attention(self, Q, K, V, mask=None):
        """Performs the full attention calculation for all n_heads heads """
        attention_vals = torch.matmul(Q, K.transpose(-2, -1))
        attention_vals = attention_vals / torch.sqrt(torch.tensor(self.dk, dtype=torch.float32))


        if mask:
            # Applies any of our 3 masks
            attention_vals = attention_vals.masked_fill(mask == 0, float('-inf'))
        
        attention_weights = torch.nn.functional.softmax(attention_vals, dim=-1)
        
        # Applies drop out for regularization then returns attention weights
        attention_weights = self.dropout(attention_weights)

        return torch.matmul(attention_weights, V)